# Phase 2: Exploratory Data Analysis & Statistical Hypothesis Testing

Conducting comprehensive univariate, bivariate, and correlation EDA alongside 3 formal statistical hypothesis tests.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.insert(0, os.path.abspath('..'))
from src.data_prep import clean_data, impute_missing_values

# Load cleaned and imputed dataset
df_raw = pd.read_csv('../data/raw/heart_disease.csv')
df = impute_missing_values(clean_data(df_raw))
print("Dataset shape:", df.shape)


## Section 1: Exploratory Data Analysis (3 Visualization Types)

### 1. Univariate Feature Distributions & Target Class Balance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Target distribution
sns.countplot(x='target', data=df, ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title("Target Class Distribution (0=Healthy, 1=Disease)")

# Plot 2: Age distribution by Target
sns.histplot(data=df, x='age', hue='target', kde=True, ax=axes[0, 1], palette='crest')
axes[0, 1].set_title("Age Distribution by Patient Outcome")

# Plot 3: Max Heart Rate (thalach) distribution
sns.kdeplot(data=df, x='thalach', hue='target', fill=True, ax=axes[1, 0], palette='flare')
axes[1, 0].set_title("Max Heart Rate (Thalach) Distribution")

# Plot 4: ST Depression (oldpeak) distribution
sns.boxplot(x='target', y='oldpeak', data=df, ax=axes[1, 1], palette='magma')
axes[1, 1].set_title("ST Depression (Oldpeak) by Patient Outcome")

plt.tight_layout()
plt.show()


### 2. Bivariate Feature Relationships & Boxplots

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='cp', y='thalach', hue='target', data=df, palette='Set2')
plt.title("Max Heart Rate across Chest Pain Types & Disease Status")
plt.xlabel("Chest Pain Type (1: Typical, 2: Atypical, 3: Non-Anginal, 4: Asymptomatic)")
plt.ylabel("Max Heart Rate (thalach)")
plt.show()


### 3. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Clinical Feature Correlation Matrix")
plt.show()


## Section 2: Statistical Hypothesis Testing (3 Hypotheses)

### Hypothesis 1: Maximum Heart Rate (`thalach`) difference between diseased and healthy patients
- **H0**: There is no difference in average max heart rate between patients with heart disease vs without.
- **H1**: Patients with heart disease have significantly lower max heart rate.

In [ ]:
group_healthy = df[df['target'] == 0]['thalach']
group_diseased = df[df['target'] == 1]['thalach']

# 1. Assumption Check: Shapiro-Wilk test for normality
sw_healthy = stats.shapiro(group_healthy)
sw_diseased = stats.shapiro(group_diseased)
print(f"Shapiro-Wilk Test (Healthy): p-value = {sw_healthy.pvalue:.4f}")
print(f"Shapiro-Wilk Test (Diseased): p-value = {sw_diseased.pvalue:.4f}")

# 2. Assumption Check: Levene's test for equal variance
lev_res = stats.levene(group_healthy, group_diseased)
print(f"Levene Variance Test: p-value = {lev_res.pvalue:.4f}")

# 3. Execution: Two-sample independent t-test (and Mann-Whitney U test fallback)
ttest_res = stats.ttest_ind(group_healthy, group_diseased)
mwu_res = stats.mannwhitneyu(group_healthy, group_diseased)

# Cohen's d Effect Size calculation
mean_diff = np.mean(group_healthy) - np.mean(group_diseased)
pooled_std = np.sqrt((np.std(group_healthy)**2 + np.std(group_diseased)**2) / 2)
cohens_d = mean_diff / pooled_std

print("\n--- HYPOTHESIS 1 RESULTS ---")
print(f"t-statistic: {ttest_res.statistic:.4f}, p-value: {ttest_res.pvalue:.4e}")
print(f"Mann-Whitney U statistic: {mwu_res.statistic:.4f}, p-value: {mwu_res.pvalue:.4e}")
print(f"Effect Size (Cohen's d): {cohens_d:.4f}")


### Hypothesis 2: Resting Blood Pressure (`trestbps`) variation across Chest Pain Types (`cp`)
- **H0**: Average blood pressure is equal across all 4 chest pain categories.
- **H1**: Blood pressure varies significantly by chest pain severity.

In [ ]:
cp_groups = [group['trestbps'].values for name, group in df.groupby('cp')]
anova_res = stats.f_oneway(*cp_groups)
kw_res = stats.kruskal(*cp_groups)

print("--- HYPOTHESIS 2 RESULTS ---")
print(f"One-Way ANOVA F-statistic: {anova_res.statistic:.4f}, p-value: {anova_res.pvalue:.4f}")
print(f"Kruskal-Wallis H-statistic: {kw_res.statistic:.4f}, p-value: {kw_res.pvalue:.4f}")


### Hypothesis 3: Association between Patient Sex (`sex`) and Disease Status (`target`)
- **H0**: Gender and heart disease occurrence are independent.
- **H1**: Gender is significantly associated with heart disease presence.

In [ ]:
contingency_table = pd.crosstab(df['sex'], df['target'])
chi2_stat, p_val, dof, expected = stats.chi2_contingency(contingency_table)

# Cramer's V effect size for categorical association
n = contingency_table.sum().sum()
min_dim = min(contingency_table.shape) - 1
cramers_v = np.sqrt(chi2_stat / (n * min_dim))

print("--- HYPOTHESIS 3 RESULTS ---")
print(f"Contingency Table:\n{contingency_table}")
print(f"Chi-Squared Statistic: {chi2_stat:.4f}, p-value: {p_val:.4e}, Degrees of Freedom: {dof}")
print(f"Effect Size (Cramer's V): {cramers_v:.4f}")
